# Prediccion de Picos de Consumo Diario mediante Regresion

Este notebook tiene como objetivo la evaluacion del modelo de regresion para predecir el consumo diario por planta y SKU, y derivar la ocurrencia de picos de consumo diario.

## Definicion de Pico
Un pico de consumo ocurre cuando el consumo real supera el umbral establecido:
Consumo Real > $umbral$ * (Forecast Mensual / 20)

## Estructura de la Evaluacion
1. Carga de datos.
2. Construccion de variables con Polars.
3. Inferencia del modelo de regresion (LightGBM Regressor).
4. Analisis de metricas de regresion (MAE, RMSE) y clasificacion derivada (Precision, Recall, F1-Score).

| Categoria | Nombre de Variable | Descripcion |
| :--- | :--- | :--- |
| **Retardos (Lags)** | `lag_1`, `lag_2`, `lag_3`, `lag_5`, `lag_10` | Consumo real registrado 1, 2, 3, 5 y 10 dias atras respecto a la fecha actual. |
| **Ventanas Moviles** | `rolling_mean_X`, `rolling_std_X`, `rolling_max_X` | Media, desviacion estandar y maximo consumo en ventanas de X dias (donde X = 3, 5 o 10), calculados sobre el consumo desfasado 1 dia para evitar filtracion de datos. |
| **Calendario** | `day_of_month`, `days_to_end_of_month` | Dia del mes actual (1 a 31) y cantidad de dias restantes para finalizar el mes en curso. |
| **Desviaciones** | `daily_forecast` | Forecast diario teorico calculado como (Forecast Mensual / 20). |
| | `cum_actual_consumption_month_lag1` | Suma acumulada de consumo real en el mes actual hasta el dia de ayer. |
| | `cum_forecast_month_lag1` | Suma acumulada de forecast esperado en el mes actual hasta el dia de ayer. |
| | `cum_deviation_month_lag1` | Desviacion acumulada acumulada hasta el dia de ayer (Consumo acumulado - Forecast acumulado). |
| **Perfil SKU** | `sku_historical_peak_rate` | Tasa historica de picos del SKU calculado en los datos de entrenamiento (Picos / Registros totales). |
| | `sku_historical_cv` | Coeficiente de variacion historico (desviacion estandar / media de consumo) para el SKU. |


In [4]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import polars as pl
import pandas as pd
import numpy as np

from src.features import build_features
from src.models import LGBMRegressor
from src.utils import transform_data, select_random_group
from src.utils import RegressionMetrics, ClassificationMetrics, ConsumptionPlotter

## 1. Carga de datos

In [ ]:
# Carga del dato crudo
data_name = "historico_consumo.parquet"
data_path = os.path.join("data", data_name) if os.path.exists(os.path.join("data", data_name)) else os.path.join("..", "data", data_name)
df_raw = pl.read_parquet(data_path)

# Transformacion a formato analitico (con la imputacion de abastecimiento integrada)
df_preparado = transform_data(df_raw)

# Guardar la serie de ejemplo separada solo para el EDA
df_eda_serie = df_preparado.filter(
    (pl.col("planta") == "SCVA") & (pl.col("sku") == "HP1_01_190_2290")
)


# Split cronologico antes de construir features para evitar leakage
fecha_corte = pl.date(2025, 10, 31)
df_train_raw = df_preparado.filter(pl.col("fecha") <= fecha_corte)
df_test_raw  = df_preparado.filter(pl.col("fecha") >  fecha_corte)

# Calcular estadisticas del SKU solo sobre train, aplicarlas al test
df_train, sku_stats = build_features(df_train_raw)
df_test,  _         = build_features(df_test_raw, sku_stats=sku_stats)

df_train.drop_nulls().head(3)


ColumnNotFoundError: unable to find column "consumo_real"; valid columns: ["planta", "sku", "Product", "fecha", "consumo_anterior_to", "stock_planta", "forecast_mensual", "target_is_peak"]

## 2. Analisis Exploratorio — SCVA / HP1_01_190_2290


In [6]:
import plotly.graph_objects as go

df_eda = df_preparado.to_pandas().sort_values("fecha")
df_eda["umbral"] = 2.0 * (df_eda["forecast_mensual"] / 20.0)
df_eda["es_pico"] = df_eda["consumo_real"] > df_eda["umbral"]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_eda["fecha"], y=df_eda["consumo_real"],
    name="Consumo Real", mode="lines",
    line=dict(color="#1f77b4", width=1.5),
))
fig.add_trace(go.Scatter(
    x=df_eda["fecha"], y=df_eda["umbral"],
    name="Umbral de Pico (2x Forecast)", mode="lines",
    line=dict(color="#d62728", width=1.5, dash="dot"),
))
fig.add_trace(go.Scatter(
    x=df_eda.loc[df_eda["es_pico"], "fecha"],
    y=df_eda.loc[df_eda["es_pico"], "consumo_real"],
    name="Pico detectado", mode="markers",
    marker=dict(color="#d62728", size=7, symbol="circle"),
))

fig.update_layout(
    title="Consumo Diario vs Umbral de Pico",
    xaxis_title="Fecha", yaxis_title="Consumo (TO)",
    template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

print(f"Tasa de picos: {df_eda['es_pico'].mean():.2%}  |  Total picos: {df_eda['es_pico'].sum()}")


Tasa de picos: 10.61%  |  Total picos: 95


## 3. Ejecucion del Modelo


In [7]:
features_calendario = [
    c for c in df_train.columns
    if c.startswith("day_of_week_") or c.startswith("month_")
]
features_base = [
    "lag_1", "lag_2", "lag_3", "lag_5", "lag_10",
    "rolling_mean_3", "rolling_std_3", "rolling_max_3",
    "rolling_mean_5", "rolling_std_5", "rolling_max_5",
    "rolling_mean_10", "rolling_std_10", "rolling_max_10",
    "day_of_month", "days_to_end_of_month",
    "daily_forecast",
    "cum_actual_consumption_month_lag1",
    "cum_forecast_month_lag1",
    "cum_deviation_month_lag1",
    "sku_historical_peak_rate",
    "sku_historical_cv",
]
feature_cols = features_base + features_calendario

model = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6)
model.fit(df_train.drop_nulls(), x_predictor=feature_cols, y="target_consumo_real")

print(model.summary())


                   Regression Metrics Summary — LGBMRegressor                   
  No. Observations : 748
  No. Features     : 39
--------------------------------------------------------------------------------
  Metric                                   Value
--------------------------------------------------------------------------------
  MAE                                     1.2369
  RMSE                                    1.7222
  WAPE                                    0.2778
  MAPE                                    0.4458
  Bias (mean residual)                    0.0000
  R-squared                               0.9388
  Adj. R-squared                          0.9354

  Dep. Variable : target_consumo_real             Date/Time: 2026-05-29 20:09:16
                        Feature Importances (Gain-based)                        
--------------------------------------------------------------------------------
 Feature                             |               Gain |  Rel. Import

In [8]:
df_test_clean  = df_test.drop_nulls()
pred_consumo   = model.predict(df_test_clean)
umbral_limite  = 2.0 * df_test_clean["daily_forecast"].to_numpy()
pred_class_pico = (pred_consumo > umbral_limite).astype(int)

df_resultado = df_test_clean.with_columns(
    pred_consumo=pl.Series(pred_consumo),
    pred_class_pico=pl.Series(pred_class_pico),
)

fig = model.plot_fit(df_resultado)
fig.show()


## 4. Validacion


In [9]:
y_true_real  = df_resultado["target_consumo_real"].to_numpy()
y_pred_real  = df_resultado["pred_consumo"].to_numpy()
y_true_class = df_resultado["target_is_peak"].to_numpy()
lag1_values  = df_resultado["lag_1"].to_numpy()

reg_metrics = RegressionMetrics(model_name="LGBMRegressor", n_features=len(feature_cols))
reg_metrics.compute(y_true_real, y_pred_real, y_baseline=lag1_values)
print(reg_metrics.summary())

scores = pred_consumo / np.where(umbral_limite == 0, 1.0, umbral_limite)

clf_metrics = ClassificationMetrics(model_name="LGBMRegressor", beta=2.0)
optimal_threshold = clf_metrics.find_optimal_threshold(y_true_class, scores, metric="fbeta")
print(f"\nOptimal threshold (F-beta): {optimal_threshold:.4f}")
print(clf_metrics.summary())


                   Regression Metrics Summary — LGBMRegressor                   
  No. Observations : 127
  No. Features     : 39
--------------------------------------------------------------------------------
  Metric                                   Value
--------------------------------------------------------------------------------
  MAE                                     5.3796
  RMSE                                    6.9053
  WAPE                                    1.2097
  MAPE                                    1.3565
  Bias (mean residual)                   -0.7835
  R-squared                              -0.2660
  Adj. R-squared                         -0.8335
--------------------------------------------------------------------------------
  Baseline (lag-1 persistence)             Value
--------------------------------------------------------------------------------
  Baseline MAE                            7.1216
  Baseline RMSE                           9.2484
  Basel